In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from typing import Dict, Tuple, List
import dotenv
import pathlib
import json
import json5
import csv
import time

dotenv.load_dotenv()

from datahub_integrations.gen_ai.description_v2 import (
    generate_entity_descriptions_for_urn,
)

from graph_helper import create_datahub_graph

In [ ]:
# Test the bedrock credentials.
from datahub_integrations.gen_ai.bedrock import call_bedrock_llm, BedrockModel


print(
    call_bedrock_llm(
        "What is the capital of France?",
        max_tokens=50,
        model=BedrockModel.CLAUDE_3_HAIKU,
    )
)

In [6]:
current_dir = pathlib.Path().resolve()
output_dir = current_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)


def write_llm_output_to_csv(llm_response, csv_path) -> None:
    with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
        csvwriter = csv.writer(csvfile)

        if len(llm_response[0]) == 4:
            csvwriter.writerow(
                ["instance", "urn", "table_description", "column_description"]
            )
        else:
            csvwriter.writerow(["instance", "urn", "raw_output"])

        for row in llm_response:
            row = list(row)
            if len(row) == 4:
                row[3] = json.dumps(row[3], indent=2)
            csvwriter.writerow(row)
    print(f"csv file {csv_path} created successfully")


def generate_entity_descriptions(
    urns_dict: Dict[str, List[str]]
) -> Tuple[List[Tuple[str, str, str, Dict[str, str]]], dict]:
    """
    This function also returns entity_infos for debugging purpose (To check the if metadata information is generated correctly) and can be removed
    """
    parsed_llm_responses = []
    entity_infos = {}
    # graph_client = create_datahub_graph("chime")
    for instance, urns in urns_dict.items():
        graph_client = create_datahub_graph(instance)
        for urn in urns:
            print("DB: ", instance, "URN: ", urn)
            try:
                result = generate_entity_descriptions_for_urn(
                    graph_client, urn
                )
            except Exception as e:
                print(f"Error generating entity descriptions for {urn}: {e}")
                continue
            entity_infos[urn] = result.extracted_entity_info
            parsed_llm_responses.append((instance, urn, result.table_description, result.column_descriptions))
    return parsed_llm_responses, entity_infos


In [ ]:
urns_dict: Dict[str, List[str]] = json5.loads(
    (current_dir / "test_urns.json").read_text()
)

exec_time = time.time()
PARSED_OUTPUT_CSV_PATH = output_dir / f"llm_output_{exec_time}_parsed.csv"
OUTPUT_CSV_PATH = output_dir / f"llm_output_{exec_time}.csv"

parsed_llm_responses, entity_infos = generate_entity_descriptions(urns_dict)
write_llm_output_to_csv(parsed_llm_responses, PARSED_OUTPUT_CSV_PATH)

In [ ]:
len(parsed_llm_responses)

In [ ]:
# Variant for testing a single urn.
urns_dict = {
    "longtailcompanions": [
        "urn:li:dataset:(urn:li:dataPlatform:dbt,long_tail_companions.adoption.adoptions,PROD)"
        
    ],
}
parsed_llm_responses, entity_infos = generate_entity_descriptions(urns_dict)
parsed_llm_responses